# Optimizacion de hiper parametros (HPO) usando Optuna y MLflow

1. Sampling Algorithms:
    1. **Tree-structured Parzen Estimator (Default)** - Bayesian optimization to efficiently search the hyperparameter space.
    2. Random Sampling - Randomly samples hyperparameter values from the search space.
    3. CMA-ES (Covariance Matrix Adaptation Evolution Strategy) - An evolutionary algorithm for difficult non-linear non-convex optimization problems.
    4. Grid Search - Exhaustively searches through a manually specified subset of the hyperparameter space.
    5. Quasi-Monte Carlo sampling - Uses low-discrepancy sequences to sample the search space more uniformly than pure random sampling.
    6. NSGA-II (Non-dominated Sorting Genetic Algorithm II) - A multi-objective optimization algorithm.
    7. Gaussian Process-based sampling (i.e. Kriging) - Uses Gaussian processes for Bayesian optimization.
    8. Optuna also allows implementing custom samplers by inheriting from the `BaseSampler` class.
2. Pruning: Optuna implements pruning algorithms to early-stop unpromising trials
3. Study Object: Users create a "study" object that manages the optimization process. The study.optimize() method is called to start the optimization, specifying the objective function and number of trials.
4. Parallel Execution: Optuna scales to parallel execution on a single node and multi-node.

<b>Definir objetivos/función de pérdida y espacio de busqueda a optimizar</b>
El espacio de busqueda es definido utilizanod funciones como: `suggest_categorical`, `suggest_float`, `suggest_int`  para el objeto Trial (ensayo) que es pasado a la función objetivo.

Documentación optuna:
* [optuna.samplers](https://optuna.readthedocs.io/en/stable/reference/samplers/index.html) for the choice of samplers
* [optuna.pruners](https://optuna.readthedocs.io/en/stable/reference/pruners.html) for the choice of pruners
* [optuna.trial.Trial](https://optuna.readthedocs.io/en/stable/reference/generated/optuna.trial.Trial.html) for a full list of functions supported to define a hyperparameter search space.

## Caracteristicas - Feacture

In [0]:
catalog = "main"
schema = "dbdemos_fs_travel"

#xp_name = f"Trabel_purchase_exp_{datetime.now().strftime('%Y-%m-%d_%H:%M:%S')}"
xp_name = f"Travel_Purchase_Exp_HPO"
xp_path = "/Workspace/Proyectos_Dev/databricks-medallion/Experiments"
model_name = "purchase_travel_classifier"
model_full_name = f"{catalog}.{schema}.{model_name}"
model_endpoint = f"{model_name}_endpoint"
model_alias = "prod"

In [0]:
try:
  from databricks.sdk import WorkspaceClient
  w = WorkspaceClient()
  r = w.workspace.mkdirs(path=xp_path)
except Exception as e:
  print(f"ERROR: couldn't create a folder for the experiment under {xp_path} - please create the folder manually or skip this init (used for job only: {e})")
  raise e

In [0]:
df = spark.table('travel_purchase').select('user_id', 'destination_id', 'purchased', 'ts').sample(fraction=0.3, seed=42)

# Split based on time to define a training and inference set (we'll do train+eval on the past & test in the most current value)
training_labels_df = df.where("ts < '2022-11-01'")
test_labels_df = df.where("ts >= '2022-11-01'")
# Set variable for label column. This will be used within the training code.
label_col = "purchased"
pos_label = "true"

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from databricks.feature_store import feature_table, FeatureLookup

fe_table_name_users = f"{catalog}.{schema}.user_features_advanced"
fe_table_name_destinations = f"{catalog}.{schema}.destination_features_advanced"

fe = FeatureEngineeringClient()

model_feature_lookups = [
      FeatureLookup(
          table_name=fe_table_name_destinations,
          lookup_key="destination_id",
          timestamp_lookup_key="ts"
      ),
      FeatureLookup(
          table_name=fe_table_name_users,
          lookup_key="user_id",
          feature_names=["mean_price_7d", "last_6m_purchases", "tenure_days", "age", "gender", "income_bracket", "loyalty_tier", "billing_state"], 
          timestamp_lookup_key="ts"
      )
]

# fe.create_training_set will look up features in model_feature_lookups with matched key from training_labels_df
training_set = fe.create_training_set(
    df=training_labels_df, # joining the original Dataset, with our FeatureLookupTable
    feature_lookups=model_feature_lookups,
    exclude_columns=["ts", "destination_id", "user_id"], # exclude id columns as we don't want them as feature
    label='purchased',
)

test_set = fe.create_training_set(
    df=test_labels_df,
    feature_lookups=model_feature_lookups,
    exclude_columns=["ts", "destination_id", "user_id"],
    label=label_col,  # keep label here for offline evaluation
)

training_pd = training_set.load_df().toPandas()
test_pd = test_set.load_df().toPandas()
X_train, y_train = training_pd.drop(label_col, axis=1), training_pd[label_col]
X_test,  y_test  = test_pd.drop(label_col, axis=1),  test_pd[label_col]

In [0]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Identify feature types
numerical_features = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X_train.select_dtypes(include=["object"]).columns.tolist()
print(f"Numerical features: {len(numerical_features)}, Categorical: {len(categorical_features)}")

numeric_transformer = Pipeline([("scaler", StandardScaler())])
categorical_transformer = Pipeline(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]
)
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

## Funciones Optuna

In [0]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from optuna.pruners import BasePruner

class ObjectiveOptuna(object):
  """
  a callable class for implementing the objective function. It takes the training dataset by a constructor's argument
  instead of loading it in each trial execution. This will speed up the execution of each trial
  """
  def __init__(self, X_train_in:pd.DataFrame, Y_train_in:pd.Series, preprocessor_in: ColumnTransformer, rng_seed:int=42, pos_label_in:str=pos_label):
    """
    X_train_in: features
    Y_train_in: label
    """

    # Set pre-processing pipeline
    self.preprocessor = preprocessor_in
    self.rng_seed = rng_seed
    self.pos_label = pos_label_in

    # Split into training and validation set
    X_train, X_val, Y_train, Y_val = train_test_split(X_train_in, Y_train_in, test_size=0.1, random_state=rng_seed)

    self.X_train = X_train
    self.Y_train = Y_train
    self.X_val = X_val
    self.Y_val = Y_val
    
  def __call__(self, trial):
    """
    Wrapper call containing data processing pipeline, training and hyperparameter tuning code.
    The function returns the weighted F1 accuracy metric to maximize in this case.
    """

    # Define list of classifiers to test
    classifier_name = trial.suggest_categorical("classifier", ["LightGBM", "LogisticRegression", "RandomForest"])
    
    if classifier_name == "LightGBM":
      # Optimize number of trees, tree depth, learning rate and maximum number of bins hyperparameters
      n_estimators = trial.suggest_int("n_estimators", 10, 200, log=True)
      max_depth = trial.suggest_int("max_depth", 3, 10)
      learning_rate = trial.suggest_float("learning_rate", 1e-2, 0.9)
      max_bin = trial.suggest_int("max_bin", 2, 256)
      num_leaves = trial.suggest_int("num_leaves", 2, 256),
      classifier_obj = LGBMClassifier(force_row_wise=True, verbose=-1,
                                      n_estimators=n_estimators, max_depth=max_depth, learning_rate=learning_rate, max_bin=max_bin, num_leaves=num_leaves, random_state=self.rng_seed)
    
    elif classifier_name == "LogisticRegression":
      # Optimize tolerance and C hyperparameters
      lr_C = trial.suggest_float("C", 1e-2, 1, log=True)
      lr_tol = trial.suggest_float('tol' , 1e-6 , 1e-3, step=1e-6)
      classifier_obj = LogisticRegression(C=lr_C, tol=lr_tol, random_state=self.rng_seed)

    elif classifier_name == "RandomForest":
      # Optimize number of trees, tree depth, min_sample split and leaf hyperparameters
      n_estimators = trial.suggest_int("n_estimators", 10, 200, log=True)
      max_depth = trial.suggest_int("max_depth", 3, 10)
      min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
      min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
      classifier_obj = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                              min_samples_split=min_samples_split, min_samples_leaf=min_samples_leaf, random_state=self.rng_seed)
            
    # Assemble the pipeline
    this_model = Pipeline(steps=[("preprocessor", self.preprocessor), ("classifier", classifier_obj)])

    # Fit the model
    mlflow.sklearn.autolog(disable=True) # Disable mlflow autologging to avoid logging artifacts for every run
    
    this_model.fit(self.X_train, self.Y_train)


    # Predict on validation set
    y_val_pred = this_model.predict(self.X_val)

    # Calculate and return F1-Score
    f1_score_binary= f1_score(self.Y_val, y_val_pred, average="binary", pos_label=self.pos_label)

    return f1_score_binary
  
class NoneValuePruner(BasePruner):
  """Custom Pruner to ignore failed trials with None value."""

  def prune(self, study, trial):
    # If the trial's value is None, prune it
    if trial.value is None:
      return True
    else:
      return False

In [0]:
optuna_sampler = optuna.samplers.TPESampler(seed=42) # Random Number Generator seed
objective_fn = ObjectiveOptuna(X_train, Y_train, preprocessor, pos_label_in=pos_label)

## Test rapido/depuracion local

In [0]:
study_debug = optuna.create_study(direction="maximize", study_name=f"{xp_name}_debug", 
                                  sampler=optuna_sampler, pruner= NoneValuePruner())
study_debug.optimize(objective_fn, n_trials=4, n_jobs=-1)

In [0]:
print("Mejor ensayo (trial):")
best_trial = study_debug.best_trial
print(f"  F1_score: {best_trial.value}")
print("  Params: ")
for key, value in best_trial.params.items():
    print(f"    {key}: {value}")

## Experimento MLflow con Optuna. Sabor: crear almacenamiento compartido para optimización distribuida
- MlflowStorage: Permite a MLflow actuar como base de datos backend para estudios Optuna
- MlflowSparkStudy: Envuelve ejecuciones en paralelo de estudios Optuna entre ejecutores Spark

In [0]:
import mlflow
from mlflow.optuna.storage import MlflowStorage
from mlflow.pyspark.optuna.study import MlflowSparkStudy

try:
  print(f"Loading experiment: {xp_name}")
  experiment_id = mlflow.get_experiment_by_name(xp_name).experiment_id
except Exception as e:
  print(f"Creating experiment: {xp_name}")
  experiment_id = mlflow.create_experiment(name=xp_name, tags={"Level":"HPO"})

mlflow.set_experiment(xp_name)
mlflow_storage = MlflowStorage(experiment_id=experiment_id)

In [0]:
study_mlflow_smoke = MlflowSparkStudy(
  pruner= NoneValuePruner(),
  sampler=optuna_sampler,
  study_name=f"{xp_name}_smoke_test",
  storage=mlflow_storage,
)
study_mlflow_smoke._directions = ["maximize"]
study_mlflow_smoke.optimize(objective_fn, n_trials=8, n_jobs=3)

## Función de entrenamiento principal
Ejecuta el experimento para entrenar el modelo utilizando Optuna para optimizar hiper parámetros. Registra el mejor modelo encontrado.

In [0]:
import warnings
import pandas as pd
from mlflow.types.utils import _infer_schema
from mlflow.exceptions import MlflowException
from mlflow.models import Model
from mlflow.pyfunc import PyFuncModel
from mlflow import pyfunc

def optuna_hpo_fn(n_trials: int, X_train: pd.DataFrame, Y_train: pd.Series, X_test: pd.DataFrame, Y_test: pd.Series, training_set_specs_in, preprocessor_in: ColumnTransformer, experiment_id: str, pos_label_in: str = pos_label, rng_seed_in: int = 2025, run_name:str = "spark-mlflow-tuning", optuna_sampler_in: optuna.samplers.TPESampler = optuna_sampler, optuna_pruner_in: optuna.pruners.BasePruner = None, n_jobs: int = 2) -> optuna.study.study.Study:
    """
    Increasing `n_jobs` may cause experiment to fail due to failed trials which return None and can't be pruned/caught in parallel mode
    """

    # Kick distributed HPO as nested runs
    objective_fn = ObjectiveOptuna(X_train, Y_train, preprocessor_in, rng_seed_in, pos_label_in)
    mlflow_optuna_study = MlflowSparkStudy(
        pruner=optuna_pruner_in,
        sampler=optuna_sampler_in,
        study_name=run_name,
        storage=mlflow_storage,
    )
    mlflow_optuna_study._directions = ["maximize"]
    mlflow_optuna_study.optimize(objective_fn, n_trials=n_trials, n_jobs=n_jobs)

    # Extract best trial info
    best_model_params = mlflow_optuna_study.best_params
    best_model_params["random_state"] = rng_seed_in
    classifier_type = best_model_params.pop('classifier')

    # Reproduce best classifier
    if classifier_type == "LightGBM":
        best_model = LGBMClassifier(force_row_wise=True, verbose=-1, **best_model_params)
    elif classifier_type  == "LogisticRegression":
        best_model = LogisticRegression(**best_model_params)
    elif classifier_type == "RandomForest":
        best_model = RandomForestClassifier(**best_model_params)

    # Enable automatic logging of input samples, metrics, parameters, and models
    mlflow.sklearn.autolog(log_input_examples=True, log_models=False, silent=True)

    active_run = mlflow.active_run()
    if not active_run:
        active_run = client.search_runs(
            experiment_ids=[experiment_id],
            filter_string=f"tags.mlflow.runName = '{run_name}'",
            order_by=["start_time DESC"],
            max_results=1)[0]
    
    run_id = active_run.info.run_id

    with mlflow.start_run(run_id=run_id, experiment_id=experiment_id) as run:
        # Fit best model and log using FE client in parent run
        model_pipeline = Pipeline(steps=[("preprocessor", objective_fn.preprocessor), ("classifier", best_model)])
        model_pipeline.fit(X_train, Y_train)

        # Evaluate model and log into experiment
        mlflow_model = Model()
        pyfunc.add_to_model(mlflow_model, loader_module="mlflow.sklearn")
        pyfunc_model = PyFuncModel(model_meta=mlflow_model, model_impl=model_pipeline)

        # Log metrics for the training set
        training_eval_result = mlflow.evaluate(
            model=pyfunc_model,
            data=X_train.assign(**{str(label_col):Y_train}),
            targets=label_col,
            model_type="classifier",
            evaluator_config = {"log_model_explainability": False,
                                "metric_prefix": "training_" , "pos_label": pos_label_in }
        )

        # Log metrics for the test set
        test_eval_result = mlflow.evaluate(
            model=pyfunc_model,
            data=X_test.assign(**{str(label_col):Y_test}),
            targets=label_col,
            model_type="classifier",
            evaluator_config = {"log_model_explainability": True,
                                "metric_prefix": "test_" , "pos_label": pos_label_in }
        )

        # Create/Log model artifacts with embedded feature lookups
        fe.log_model(
            model=model_pipeline,
            artifact_path="model",
            flavor=mlflow.sklearn,
            training_set=training_set_specs_in,
            # env_manager="uv"
        )
        mlflow.end_run()

    return mlflow_optuna_study

## Ejecutar entrenamientos

In [0]:
# Invoke training function which will be distributed accross workers/nodes
distributed_study = optuna_hpo_fn(
  n_trials=32, # Decrease this for live demo purposes
  X_train=X_train,
  X_test=X_test,
  Y_train=Y_train,
  Y_test=Y_test,
  training_set_specs_in=training_set_specs,
  preprocessor_in=preprocessor,
  experiment_id=experiment_id,
  pos_label_in=pos_label,
  rng_seed_in=42,
  run_name="mlops-hpo-best-run", # "smoke-test"
  optuna_sampler_in=optuna_sampler,
  optuna_pruner_in=NoneValuePruner(),
  n_jobs = 2, # Increase this to number for more parallel trials
)